In [1]:
import math
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

生成数据集

In [2]:
max_degree = 20  # 多项式的最大阶数
n_train, n_test = 100, 100  # 训练和测试数据集大小
true_w = np.zeros(max_degree)  # 分配大量的空间
true_w[0:4] = np.array([5, 1.2, -3.4, 5.6])

features = np.random.normal(size=(n_train + n_test, 1))
np.random.shuffle(features)
poly_features = np.power(features, np.arange(max_degree).reshape(1, -1))
for i in range(max_degree):
    poly_features[:, i] /= math.gamma(i + 1)  # gamma(n)=(n-1)!
# labels的维度:(n_train+n_test,)
labels = np.dot(poly_features, true_w)
labels += np.random.normal(scale=0.1, size=labels.shape)

In [3]:
features[:2], poly_features[:2, :], labels[:2]

(array([[-0.69688201],
        [-0.54053365]]),
 array([[ 1.00000000e+00, -6.96882006e-01,  2.42822265e-01,
         -5.64061557e-02,  9.82710873e-03, -1.36966705e-03,
          1.59082720e-04, -1.58374121e-05,  1.37960094e-06,
         -1.06824341e-07,  7.44439612e-09, -4.71624155e-10,
          2.73888656e-11, -1.46821597e-12,  7.30838063e-14,
         -3.39538597e-15,  1.47886461e-16, -6.06231846e-18,
          2.34706703e-19, -8.60857252e-21],
        [ 1.00000000e+00, -5.40533652e-01,  1.46088314e-01,
         -2.63218833e-02,  3.55696593e-03, -3.84531957e-04,
          3.46420771e-05, -2.67502978e-06,  1.80742952e-07,
         -1.08552942e-08,  5.86765181e-10, -2.88333024e-11,
          1.29878085e-12, -5.40026736e-14,  2.08501874e-15,
         -7.51348528e-17,  2.53830727e-18, -8.07082647e-20,
          2.42364072e-21, -6.89504932e-23]]),
 array([3.00633218, 3.72316374]))

模型评估

In [4]:
def evaluate_loss(net, data_iter, loss):
    """评估给定数据集上模型的损失"""
    total_loss, total_count = 0.0, 0
    net.eval()
    with torch.no_grad():
        for X, y in data_iter:
            out = net(X)
            y = y.reshape(out.shape)
            l = loss(out, y)
            total_loss += l.sum().item()
            total_count += l.numel()
    net.train()
    return total_loss / total_count

In [5]:
def train(train_features, test_features, train_labels, test_labels, num_epochs=400):
    loss = nn.MSELoss(reduction='none')
    input_shape = train_features.shape[-1]
    # 不设置偏置，因为我们已经在多项式中实现了它
    net = nn.Sequential(nn.Linear(input_shape, 1, bias=False))

    train_X = torch.tensor(train_features, dtype=torch.float32)
    train_y = torch.tensor(train_labels, dtype=torch.float32).reshape(-1, 1)
    test_X = torch.tensor(test_features, dtype=torch.float32)
    test_y = torch.tensor(test_labels, dtype=torch.float32).reshape(-1, 1)

    batch_size = min(10, train_y.shape[0])
    train_iter = DataLoader(TensorDataset(train_X, train_y), batch_size=batch_size, shuffle=True)
    test_iter = DataLoader(TensorDataset(test_X, test_y), batch_size=batch_size, shuffle=False)

    trainer = torch.optim.SGD(net.parameters(), lr=0.01)

    for epoch in range(num_epochs):
        for X, y in train_iter:
            trainer.zero_grad()
            out = net(X)
            l = loss(out, y)
            l.mean().backward()
            trainer.step()

        if epoch == 0 or (epoch + 1) % 20 == 0:
            train_l = evaluate_loss(net, train_iter, loss)
            test_l = evaluate_loss(net, test_iter, loss)
            print(f"epoch {epoch + 1:4d}: train loss={train_l:.6f}, test loss={test_l:.6f}")

    print('weight:', net[0].weight.data.numpy())

### 正常

In [6]:
# 从多项式特征中选择前4个维度，即1,x,x^2/2!,x^3/3!
train(poly_features[:n_train, :4], poly_features[n_train:, :4],
    labels[:n_train], labels[n_train:])

epoch    1: train loss=20.981712, test loss=15.227650
epoch   20: train loss=1.301263, test loss=0.635550
epoch   40: train loss=0.331489, test loss=0.450834
epoch   60: train loss=0.167979, test loss=0.450521
epoch   80: train loss=0.116034, test loss=0.382286
epoch  100: train loss=0.086341, test loss=0.298635
epoch  120: train loss=0.065612, test loss=0.225575
epoch  140: train loss=0.050532, test loss=0.167537
epoch  160: train loss=0.039510, test loss=0.124827
epoch  180: train loss=0.031434, test loss=0.092784
epoch  200: train loss=0.025512, test loss=0.069411
epoch  220: train loss=0.021176, test loss=0.052404
epoch  240: train loss=0.018002, test loss=0.040160
epoch  260: train loss=0.015675, test loss=0.031114
epoch  280: train loss=0.013971, test loss=0.024569
epoch  300: train loss=0.012721, test loss=0.019860
epoch  320: train loss=0.011807, test loss=0.016448
epoch  340: train loss=0.011136, test loss=0.013983
epoch  360: train loss=0.010645, test loss=0.012232
epoch  380

### 欠拟合

In [7]:
# 从多项式特征中选择前2个维度，即1和x
train(poly_features[:n_train, :2], poly_features[n_train:, :2],
    labels[:n_train], labels[n_train:])

epoch    1: train loss=26.111031, test loss=18.431485
epoch   20: train loss=9.044723, test loss=3.037738
epoch   40: train loss=9.030907, test loss=3.186422
epoch   60: train loss=9.030763, test loss=3.178720
epoch   80: train loss=9.030574, test loss=3.153503
epoch  100: train loss=9.030979, test loss=3.187699
epoch  120: train loss=9.030591, test loss=3.144653
epoch  140: train loss=9.030569, test loss=3.154288
epoch  160: train loss=9.030995, test loss=3.190183
epoch  180: train loss=9.030581, test loss=3.163474
epoch  200: train loss=9.030558, test loss=3.154856
epoch  220: train loss=9.030751, test loss=3.176825
epoch  240: train loss=9.030621, test loss=3.164099
epoch  260: train loss=9.030568, test loss=3.161066
epoch  280: train loss=9.030788, test loss=3.129465
epoch  300: train loss=9.030608, test loss=3.146224
epoch  320: train loss=9.031218, test loss=3.197595
epoch  340: train loss=9.030755, test loss=3.131598
epoch  360: train loss=9.030866, test loss=3.183141
epoch  380

### 过拟合

In [8]:
# 从多项式特征中选取所有维度
train(poly_features[:n_train, :], poly_features[n_train:, :],
    labels[:n_train], labels[n_train:], num_epochs=1500)

epoch    1: train loss=24.074799, test loss=17.409208
epoch   20: train loss=0.971804, test loss=0.682981
epoch   40: train loss=0.194506, test loss=0.677529
epoch   60: train loss=0.101580, test loss=0.738999
epoch   80: train loss=0.078809, test loss=0.689449
epoch  100: train loss=0.066674, test loss=0.613613
epoch  120: train loss=0.058310, test loss=0.542886
epoch  140: train loss=0.052235, test loss=0.482565
epoch  160: train loss=0.047720, test loss=0.432869
epoch  180: train loss=0.044291, test loss=0.391052
epoch  200: train loss=0.041627, test loss=0.357432
epoch  220: train loss=0.039506, test loss=0.328766
epoch  240: train loss=0.037771, test loss=0.304154
epoch  260: train loss=0.036312, test loss=0.284152
epoch  280: train loss=0.035054, test loss=0.265599
epoch  300: train loss=0.033942, test loss=0.250635
epoch  320: train loss=0.032943, test loss=0.236550
epoch  340: train loss=0.032032, test loss=0.224979
epoch  360: train loss=0.031183, test loss=0.213426
epoch  380